In [0]:
isplay(
    dbutils.fs.ls(
        "abfss://gold@adfassignment7.dfs.core.windows.net/"
    )
)

In [0]:
# Step 2: Read Product Delta table from Silver

product_path = "abfss://silver@adfassignment7.dfs.core.windows.net/sales_view/product/"

product_df = spark.read \
    .format("delta") \
    .load(product_path)

display(product_df)

In [0]:
#  Read Store Delta table from Silver

store_path = "abfss://silver@adfassignment7.dfs.core.windows.net/sales_view/store/"

store_df = spark.read \
    .format("delta") \
    .load(store_path)

display(store_df)

In [0]:
#  Join Product and Store

from pyspark.sql.functions import col

product_store_df = product_df.alias("p") \
    .join(
        store_df.alias("s"),
        col("p.store_id") == col("s.store_id"),
        "left"
    )

display(product_store_df)

In [0]:
# Correct Product ID column name

product_df = product_df.withColumnRenamed(
    "product__i_d",
    "product_id"
)

display(product_df)

In [0]:
#  Join Product and Store again

product_store_df = product_df.alias("p") \
    .join(
        store_df.alias("s"),
        col("p.store_id") == col("s.store_id"),
        "left"
    )

display(product_store_df)

In [0]:
#  Select required Product + Store columns

product_store_df = product_store_df.select(
    col("s.store_id").alias("store_id"),
    col("s.store_name").alias("store_name"),
    col("s.location").alias("location"),
    col("s.manager_name").alias("manager_name"),

    col("p.product_id").alias("product_id"),
    col("p.product_name").alias("product_name"),
    col("p.product_code").alias("product_code"),
    col("p.description").alias("description"),
    col("p.category_id").alias("category_id"),
    col("p.price").alias("price"),
    col("p.stock_quantity").alias("stock_quantity"),
    col("p.supplier_id").alias("supplier_id"),
    col("p.created_at").alias("product_created_at"),
    col("p.updated_at").alias("product_updated_at"),
    col("p.image_url").alias("image_url"),
    col("p.weight").alias("weight"),
    col("p.expiry_date").alias("expiry_date"),
    col("p.is_active").alias("is_active"),
    col("p.tax_rate").alias("tax_rate")
)

display(product_store_df)

In [0]:
#Read Customer Sales Delta table from Silver

customer_sales_path = "abfss://silver@adfassignment7.dfs.core.windows.net/sales_view/customer_sales/"

customer_sales_df = spark.read \
    .format("delta") \
    .load(customer_sales_path)

display(customer_sales_df)

In [0]:
#  Correct Customer Sales Product ID

customer_sales_df = customer_sales_df.withColumnRenamed(
    "product__id",
    "product_id"
)

display(customer_sales_df)

In [0]:
#  Join Customer Sales with Product + Store

final_df = customer_sales_df.alias("cs") \
    .join(
        product_store_df.alias("ps"),
        col("cs.product_id") == col("ps.product_id"),
        "left"
    )

display(final_df)

In [0]:
display(
    customer_sales_df.limit(5)
)

In [0]:
print(product_store_df.columns)

In [0]:
# Join Customer Sales with Product + Store

from pyspark.sql.functions import col

final_df = customer_sales_df.alias("cs") \
    .join(
        product_store_df.alias("ps"),
        col("cs.product_id") == col("ps.product_id"),
        "left"
    )

display(final_df)

In [0]:
#  Select final Gold columns

final_df = final_df.select(
    col("cs.order_date").alias("OrderDate"),
    col("cs.category").alias("Category"),
    col("cs.city").alias("City"),
    col("cs.customer_i_d").alias("CustomerID"),
    col("cs.order_i_d").alias("OrderID"),
    col("cs.product_id").alias("Product ID"),
    col("cs.profit").alias("Profit"),
    col("cs.region").alias("Region"),
    col("cs.sales").alias("Sales"),
    col("cs.segment").alias("Segment"),
    col("cs.ship_date").alias("ShipDate"),
    col("cs.ship_mode").alias("ShipMode"),
    col("cs.latitude").alias("latitude"),
    col("cs.longitude").alias("longitude"),
    col("ps.store_name").alias("store_name"),
    col("ps.location").alias("location"),
    col("ps.manager_name").alias("manager_name"),
    col("ps.product_name").alias("product_name"),
    col("ps.price").alias("price"),
    col("ps.stock_quantity").alias("stock_quantity"),
    col("ps.image_url").alias("image_url")
)

display(final_df)

In [0]:
#Write StoreProductSalesAnalysis to Gold

gold_path = "abfss://gold@adfassignment7.dfs.core.windows.net/sales_view/StoreProductSalesAnalysis/"

final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(gold_path)

print("StoreProductSalesAnalysis written successfully to Gold")

In [0]:
final_df = final_df.withColumnRenamed(
    "Product ID",
    "Product_ID"
)

display(final_df)

In [0]:
#  Read Gold Delta table

gold_df = spark.read \
    .format("delta") \
    .load(gold_path)

display(gold_df)